# 🐧 เทรน G1 บน Ubuntu + NVIDIA GPU (GTX 1060) — ได้โมเดล .pt

notebook นี้เทรน G1 ให้เดิน โดย**ใช้ GPU จริง** เร็วกว่าเทรนบน CPU มาก
ปรับพารามิเตอร์มาให้เหมาะกับ **GTX 1060 (VRAM 6GB)** โดยเฉพาะ

**ต่างจาก `train_g1_mac.ipynb`:** อันนั้นบังคับ CPU, อันนี้ใช้ GPU +
ปรับ `num-envs` ให้พอดี VRAM 6GB (ไม่งั้น out of memory)

**⚠️ เตรียมเครื่องก่อน (รันใน terminal ครั้งเดียว):**
```bash
git clone https://github.com/anunpanya9/mjlab-custom.git
cd mjlab-custom
uv sync            # ติดตั้ง mjlab + torch(CUDA) อัตโนมัติ
uv add --dev ipykernel
```
จากนั้นเปิด notebook นี้ แล้วเลือก kernel เป็น **`.venv`** ของโปรเจค

**หมายเหตุ GTX 1060:** เป็นการ์ดปี 2016 (Pascal) — เทรนได้แต่ช้ากว่าการ์ด
ใหม่. ต้องมี **NVIDIA driver + CUDA 12.4+** ติดตั้งไว้

## 1) เช็ค GPU + VRAM

ยืนยันว่า PyTorch เห็น GPU และดูว่ามี VRAM เท่าไร (ใช้กำหนด num-envs)

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'PyTorch ไม่เห็น GPU! เช็ค: (1) nvidia-smi ใช้ได้ไหม '
    '(2) uv sync ลง torch เวอร์ชัน CUDA แล้วหรือยัง'
)
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU        : {name}')
print(f'VRAM       : {vram_gb:.1f} GB')
print(f'CUDA cap.  : {torch.cuda.get_device_capability(0)}')
print(f'torch CUDA : {torch.version.cuda}')

## 2) เลือกจำนวน env ให้พอดี VRAM

**แนวคิด:** ยิ่ง env เยอะยิ่งเทรนเร็ว แต่กิน VRAM มากขึ้น. default ของ mjlab
(4096) จะ **OOM บน 1060**. เราเลือกค่าที่ปลอดภัยตาม VRAM:
- VRAM ≥ 6GB → 1024 env
- VRAM ~3-4GB → 512 env

ถ้าเทรนแล้วเจอ `CUDA out of memory` ให้ลดค่านี้ลงครึ่งหนึ่ง

In [ ]:
if vram_gb >= 5.5:
    num_envs = 1024
elif vram_gb >= 3.0:
    num_envs = 512
else:
    num_envs = 256
print(f'เลือก num_envs = {num_envs} (สำหรับ VRAM {vram_gb:.1f} GB)')

## 3) เทรน! (ใช้ GPU)

เรียก train script — ไม่ตั้ง `--gpu-ids` mjlab จะใช้ GPU ที่เห็นอัตโนมัติ
- `--agent.max-iterations 500` — บน 1060 เร็วกว่า CPU มาก (~500 รอบพอเห็น
  หุ่นเริ่มเดินเป็นรูปเป็นร่าง). เพิ่มเป็น 3000+ ถ้าอยากได้ผลสวย
- `--agent.logger tensorboard` — เลี่ยง wandb login

**ดู `Mean reward` เพิ่มขึ้นเรื่อยๆ = policy กำลังเรียนรู้**

> รอบแรกช้าเพราะ MuJoCo Warp compile CUDA kernel — ปกติ

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cmd = [
    sys.executable, '-m', 'mjlab.scripts.train',
    'Mjlab-Velocity-Flat-Unitree-G1',
    '--env.scene.num-envs', str(num_envs),
    '--agent.max-iterations', '500',
    '--agent.save-interval', '50',
    '--agent.logger', 'tensorboard',
]
print('รัน:', ' '.join(cmd[2:]))
print('=' * 60)
proc = subprocess.Popen(
    cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        if any(k in line for k in ('Mean reward', 'Learning iteration',
                                   'ETA', 'Storing', 'out of memory',
                                   'error', 'Error', 'Traceback')):
            print(line.rstrip())
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()  # กด stop -> ฆ่า subprocess ให้จริง
    print('หยุดการเทรนแล้ว')
print('=' * 60, '| exit code =', proc.returncode)

## 4) หาไฟล์โมเดล .pt

In [ ]:
log_dir = REPO / 'logs' / 'rsl_rl' / 'g1_velocity'
runs = sorted(log_dir.glob('*'), key=os.path.getmtime, reverse=True)
assert runs, 'ไม่พบ run — เทรนสำเร็จหรือยัง?'
latest = runs[0]
ckpts = sorted(latest.glob('model_*.pt'),
               key=lambda p: int(''.join(filter(str.isdigit, p.stem))))
checkpoint = str(ckpts[-1])
print('✅ โมเดล:', checkpoint)
print('   ขนาด:', round(Path(checkpoint).stat().st_size / 1e6, 1), 'MB')

## 5) ทดสอบ — เรนเดอร์วิดีโอให้ policy สั่งหุ่นเดิน

โหลด checkpoint แล้วให้ policy สั่งหุ่นจริง อัดเป็นวิดีโอ. ถ้าเครื่องไม่มีจอ
(รันผ่าน SSH) ตั้ง `MUJOCO_GL=egl` ก่อน import

In [ ]:
os.environ.setdefault('MUJOCO_GL', 'egl')  # headless (ถ้ารันผ่าน SSH)
import numpy as np, imageio
from dataclasses import asdict
import mjlab.tasks  # noqa: F401
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner

TASK = 'Mjlab-Velocity-Flat-Unitree-G1'
device = 'cuda'
env_cfg = load_env_cfg(TASK, play=True)
env_cfg.scene.num_envs = 1
eval_env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode='rgb_array')
agent_cfg = load_rl_cfg(TASK)
runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)
runner = runner_cls(wrapped, asdict(agent_cfg), device=device)
runner.load(checkpoint, load_cfg={'actor': True}, strict=True, map_location=device)
policy = runner.get_inference_policy(device=device)

obs = wrapped.get_observations()
frames = []
for step in range(200):
    with torch.inference_mode():
        action = policy(obs)
    obs, _, _, _ = wrapped.step(action)
    frames.append(eval_env.render())
out = str(REPO / 'g1_trained_ubuntu.mp4')
imageio.mimsave(out, frames, fps=30)
print('✓ วิดีโอ:', out)

In [ ]:
from IPython.display import Video
Video(out, embed=True, width=480)

## 6) เอาโมเดลไปใช้ + ปรับจูน

**เล่นดู interactive** (viser viewer บนเบราว์เซอร์):
```bash
uv run play Mjlab-Velocity-Flat-Unitree-G1 \
    --checkpoint-file <path จากขั้น 4> --viewer viser --num-envs 1
```

**เดินยังไม่สวย?** เพิ่ม `--agent.max-iterations` เป็น 3000-10000 (บน 1060
ใช้เวลาหลายสิบนาที-ชั่วโมง แต่ผลดีขึ้นชัด)

**CUDA out of memory?** ลด `num_envs` (cell ขั้น 2) ลงครึ่งหนึ่งแล้วรันใหม่

**อยากเทรนงานหยิบของ?** เปลี่ยน task เป็น `Mjlab-Lift-Cube-G1` และ path
เป็น `g1_lift_cube`. งานหยิบมีมือ Dex3 (contact เยอะ) กิน VRAM มากกว่า —
อาจต้องลด num_envs ลงอีก